# Study 959 — Crypto Fee War — the teardown

Three tracking-difference estimators and where the naive one lies; the fund-versus-spot measurement floor that makes the cross-sectional instrument necessary; the HAC *t* and moving-block bootstrap on the cheapest-minus-priciest spread; a Spearman rank test reported against its **attainable** critical value; the waiver event study; the excess-of-cash ownership race with one execution lag; the borrow and tax sweeps; and the live synthetic control.

Cohort: the ten US spot-bitcoin ETFs launched 2024-01-11, vs **BTC-USD**. Window 2024-01-11 → 2026-06-30, 618 sessions, 29 complete months. Every real number is frozen from `docs/results.md` (fingerprint `8463d167f5aa`, as-of 2026-06-30).

In [1]:
R = {'start': '2024-01-11', 'end': '2026-06-30', 'n_days': 618, 'n_months': 29, 'fp': '8463d167f5aa', 'sd_vs_bench': 134.7, 'sd_vs_peer': 8.9, 'floor_ratio': 15.2, 'det_vs_bench': 481.0, 'det_vs_peer': 66.0, 'cheap': 'EZBC', 'dear': 'GBTC', 'spread': 145.8, 'spread_t': 8.43, 'pos_months': 26, 'ci_lo': 110.5, 'ci_hi': 187.0, 'frac_neg': 0.0, 'sd_month': 12.4, 'sd_daily': 10.2, 'vs_gbtc': {'IBIT': (133.6, 7.04, 24), 'FBTC': (136.3, 8.23, 25), 'ARKB': (139.9, 7.31, 25), 'BITB': (140.1, 9.77, 27), 'HODL': (161.7, 10.56, 28), 'BRRR': (140.7, 9.36, 26), 'BTCO': (147.2, 5.8, 24), 'EZBC': (145.8, 8.43, 26), 'BTCW': (139.8, 6.76, 24)}, 'track': {'IBIT': (25, -41.3, -25.0, -5.16, -15.7, -0.13, 5.1, 0.38), 'FBTC': (25, -45.6, -20.5, -4.28, -13.1, -0.12, 7.8, 1.04), 'ARKB': (21, -46.5, -20.8, -4.37, -9.5, -0.08, 11.4, 1.1), 'BITB': (20, -49.9, -18.0, -3.7, -9.3, -0.09, 11.6, 1.88), 'HODL': (20, -23.9, 0.2, 0.05, 12.3, 0.11, 33.2, 4.1), 'BRRR': (25, -56.9, -21.1, -4.4, -8.7, -0.08, 12.2, 1.63), 'BTCO': (25, -25.1, -19.4, -4.12, -2.2, -0.02, 18.7, 1.29), 'EZBC': (19, -29.9, -15.5, -3.33, -3.6, -0.03, 17.3, 1.66), 'BTCW': (25, -19.1, -17.2, -3.62, -9.6, -0.09, 11.3, 0.74), 'GBTC': (150, -79.4, -156.8, -33.22, -149.4, -1.39, -128.5, -9.57)}, 'trim': {1: (141.7, 7.78, 27), 2: (140.5, 7.18, 25), 3: (135.6, 6.06, 23)}, 'slope_all': 141.3, 'slope_acf1': 0.8, 'slope_nine_med': 140.1, 'slope_nine_lo': 131.8, 'slope_nine_hi': 157.1, 'era_e_n': 11, 'era_e': 186.5, 'era_e_t': 5.76, 'era_e_pos': 9, 'era_l_n': 18, 'era_l': 120.9, 'era_l_t': 9.1, 'era_l_pos': 17, 'rank': {'headline / all ten': (-0.642, 0.0514, 0.642, -1.124, 0.978), 'headline / cheap nine': (-0.495, 0.1792, 0.688, -1.522, 0.247), 'blended / all ten': (-0.498, 0.1472, 0.644, -1.083, 0.985), 'blended / cheap nine': (-0.31, 0.4076, 0.678, -1.645, 0.552)}, 'tier_spread': 6.0, 'waiver': {'IBIT': ('2025-01-10', -20.9, 21.1, 42.0, 0.73, -13), 'FBTC': ('2024-07-31', 18.3, 5.0, -13.3, -0.3, -25), 'ARKB': ('2024-07-11', 34.6, 6.6, -28.0, -0.22, -21), 'BITB': ('2024-07-11', 4.0, 13.2, 9.2, 0.18, -20), 'HODL': ('2025-03-31', 28.9, 37.2, 8.3, 0.25, -20), 'BTCO': ('2024-07-11', -23.8, 27.5, 51.2, 0.47, -25), 'EZBC': ('2024-08-02', 26.1, 15.0, -11.1, -0.11, -19), 'BTCW': ('2024-07-11', 19.0, 9.6, -9.3, -0.17, -25)}, 'mini_months': 23, 'mini_vs_gbtc': 130.6, 'mini_vs_gbtc_t': 2.84, 'mini_vs_gbtc_pos': 20, 'mini_vs_cheap': -0.0, 'mini_vs_cheap_t': -0.0, 'mini_fee_gap': 135, 'race_n': 617, 'race_switches': 7, 'sh_cheap': 0.3467, 'cagr_cheap': 9.67, 'tot_cheap': 25.36, 'sh_dear': 0.3365, 'cagr_dear': 9.13, 'tot_dear': 23.84, 'sh_rot': 0.3457, 'cagr_rot': 9.64, 'tot_rot': 25.28, 'sharpe_gap': 0.0102, 'vol_ann': 49.9, 'be_2bp': 10.0, 'be_5bp': 25.0, 'be_25bp': 125.2, 'tax_20_100': 7.2, 'tax_20_300': 11.1, 'tax_238_300': 13.5, 'ls_gross': 137.8, 'ls_dead_at': 150.0, 'syn_pl_pair': 124.4, 'syn_pl_planted': 131.0, 'syn_pl_t': 6.55, 'syn_pl_slope': -0.97, 'syn_pl_r2': 0.995, 'syn_nl_pair': -7.9, 'syn_nl_t': -0.44, 'syn_nl_slope': 0.039, 'syn_null_seeds': 120, 'syn_null_rho': 0.016, 'syn_null_fire_pct': 4.2, 'syn_null_max_t': 0.92, 'syn_pow_seeds': 24, 'syn_pow_t': 7.0, 'syn_pow_slope': -1.01, 'syn_pow_slope_sd': 0.045, 'syn_pow_rank_fire': 9}

## 1. The measurement floor decides the estimator

BTC-USD is a 24/7 quote; the wrappers strike at 16:00 New York. The fund-minus-benchmark difference therefore carries a mean-zero **clock stub**; the fund-minus-fund difference does not, because the stub is common.

In [2]:
print(f"daily sd, fund - BTC-USD : {R['sd_vs_bench']:7.1f} bp")
print(f"daily sd, fund - peer    : {R['sd_vs_peer']:7.1f} bp   (ratio {R['floor_ratio']:.1f}x)")
print()
print(f"smallest |t|=2-detectable gap on {R['n_months']} months:")
print(f"  vs BTC-USD : {R['det_vs_bench']:5.0f} bp/yr")
print(f"  vs a peer  : {R['det_vs_peer']:5.0f} bp/yr")
print(f"a 20 bp/yr fee = {20/252:.3f} bp/day.")

daily sd, fund - BTC-USD :   134.7 bp
daily sd, fund - peer    :     8.9 bp   (ratio 15.2x)

smallest |t|=2-detectable gap on 29 months:
  vs BTC-USD :   481 bp/yr
  vs a peer  :    66 bp/yr
a 20 bp/yr fee = 0.079 bp/day.


> 💡 **In plain words.** Bitcoin keeps moving after the funds close, so measuring a fund against bitcoin is like weighing a letter on a bathroom scale during an earthquake. Weigh two letters against each other and the earthquake cancels.

## 2. Three estimators of one quantity

`endpoint` (two anchors, annualised), `trend slope` (OLS of the log ratio on time, HAC se, 618 anchors), `monthly` (mean non-overlapping complete-calendar-month difference, HAC *t*), plus the **cohort-relative** monthly figure in which the benchmark's stub cancels entirely.

In [3]:
hdr = ('fund', 'fee', 'endpt', 'slope', 't', 'vsBTC', 't', 'vsPack', 't')
print('%-6s %5s %8s %8s %7s %9s %7s %9s %7s' % hdr)
for tk, v in R['track'].items():
    print('%-6s %5d %8.1f %8.1f %7.2f %9.1f %7.2f %9.1f %7.2f'
          % (tk, v[0], v[1], v[2], v[3], v[4], v[5], v[6], v[7]))

fund     fee    endpt    slope       t     vsBTC       t    vsPack       t
IBIT      25    -41.3    -25.0   -5.16     -15.7   -0.13       5.1    0.38
FBTC      25    -45.6    -20.5   -4.28     -13.1   -0.12       7.8    1.04
ARKB      21    -46.5    -20.8   -4.37      -9.5   -0.08      11.4    1.10
BITB      20    -49.9    -18.0   -3.70      -9.3   -0.09      11.6    1.88
HODL      20    -23.9      0.2    0.05      12.3    0.11      33.2    4.10
BRRR      25    -56.9    -21.1   -4.40      -8.7   -0.08      12.2    1.63
BTCO      25    -25.1    -19.4   -4.12      -2.2   -0.02      18.7    1.29
EZBC      19    -29.9    -15.5   -3.33      -3.6   -0.03      17.3    1.66
BTCW      25    -19.1    -17.2   -3.62      -9.6   -0.09      11.3    0.74
GBTC     150    -79.4   -156.8  -33.22    -149.4   -1.39    -128.5   -9.57


Three things to read off this table.

1. **The endpoint estimator lies by 70 bp/yr.** It puts GBTC's leak at -79.4 bp/yr — roughly half the truth — because 2024-01-11 was GBTC's conversion day and its closing discount that afternoon is baked into the first anchor. Two anchors, one dislocated, wrong answer.
2. **The vs-BTC column is unusable.** GBTC's -149.4 bp/yr — the one unambiguously correct number in it — carries *t* = -1.39. The floor is 481 bp/yr.
3. **The trend-slope column looks sharp and is over-confident.** Its residual is a persistent AR(1) premium/discount, so the HAC correction is not enough; it is quoted, never leaned on. Every headline is the monthly estimator.

> 💡 **In plain words.** Measure a slow leak by comparing the level on two particular days and you are at the mercy of what happened on those two days. Measuring month by month does **not** make that go away — log increments telescope, so the monthly mean is still an endpoint estimate, just between two *better* days (the contaminated conversion close is gone). What it genuinely adds is a dispersion: 29 separate signs, and a *t*. The anchors themselves are checked in §3b.

**HODL is the exception worth naming:** +33.2 bp/yr against the pack at *t* = +4.10 — beating the cohort by more than its own 20 bp fee. A wrapper cannot out-earn the coin it holds; the residual is a quoting artefact plus the cohort's longest waiver (scheduled to 2025-03-31). Named, not promoted to a finding.

## 3. The headline — a pure-tape pair spread

Fund-minus-fund monthly tracking difference, annualised. Uses **no fee input**. HAC *t* at 3 monthly lags; moving-block bootstrap (5,000 draws, 3-month blocks).

In [4]:
print(f"{R['cheap']} - {R['dear']}: {R['spread']:+.1f} bp/yr   HAC t = {R['spread_t']:+.2f}")
print(f"  positive in {R['pos_months']}/{R['n_months']} months")
print(f"  block-bootstrap 95% CI [{R['ci_lo']:+.1f}, {R['ci_hi']:+.1f}] bp/yr, "
      f"share of resamples < 0: {R['frac_neg']:.4f}")
print(f"  monthly sd {R['sd_month']:.1f} bp | daily sd {R['sd_daily']:.1f} bp")
print()
print('every cheap wrapper against GBTC (nine independent cheap legs):')
for tk, (s, t, p) in R['vs_gbtc'].items():
    print(f'  {tk:6s} {s:+8.1f} bp/yr   t={t:+6.2f}   {p:2d}/{R["n_months"]}')
vals = [v[0] for v in R['vs_gbtc'].values()]
print(f'\n  dispersion across the nine: {min(vals):.1f} to {max(vals):.1f} bp/yr'
      f'  (fee gap 150-20 = 130)')

EZBC - GBTC: +145.8 bp/yr   HAC t = +8.43
  positive in 26/29 months
  block-bootstrap 95% CI [+110.5, +187.0] bp/yr, share of resamples < 0: 0.0000
  monthly sd 12.4 bp | daily sd 10.2 bp

every cheap wrapper against GBTC (nine independent cheap legs):
  IBIT     +133.6 bp/yr   t= +7.04   24/29
  FBTC     +136.3 bp/yr   t= +8.23   25/29
  ARKB     +139.9 bp/yr   t= +7.31   25/29
  BITB     +140.1 bp/yr   t= +9.77   27/29
  HODL     +161.7 bp/yr   t=+10.56   28/29
  BRRR     +140.7 bp/yr   t= +9.36   26/29
  BTCO     +147.2 bp/yr   t= +5.80   24/29
  EZBC     +145.8 bp/yr   t= +8.43   26/29
  BTCW     +139.8 bp/yr   t= +6.76   24/29

  dispersion across the nine: 133.6 to 161.7 bp/yr  (fee gap 150-20 = 130)


## 3b. Is the headline an artefact of its own two anchors?

The monthly mean telescopes, so it *is* an endpoint estimate between the first and last complete month-end. So test those anchors: trim months off **both** ends, and re-estimate with every session as an anchor (the OLS trend slope).

In [5]:
print('%-38s %10s %8s' % ('estimator', 'bp/yr', 'HAC t'))
print('%-38s %+10.1f %+8.2f' % ('headline (29 months)', R['spread'], R['spread_t']))
for k, (v, t, n) in R['trim'].items():
    print('%-38s %+10.1f %+8.2f' % (f'trim {k} month(s) off both ends (n={n})', v, t))
print('%-38s %+10.1f %8s' % ('all-618-anchor OLS trend slope', R['slope_all'], 'n/a'))
print('%-38s %+10.1f %8s' % ('same slope, median of the nine legs', R['slope_nine_med'], 'n/a'))
print(f"\nevery anchor-free version returns 135-142 bp/yr; the headline is ~5 bp rich.")

estimator                                   bp/yr    HAC t
headline (29 months)                       +145.8    +8.43
trim 1 month(s) off both ends (n=27)       +141.7    +7.78
trim 2 month(s) off both ends (n=25)       +140.5    +7.18
trim 3 month(s) off both ends (n=23)       +135.6    +6.06
all-618-anchor OLS trend slope             +141.3      n/a
same slope, median of the nine legs        +140.1      n/a

every anchor-free version returns 135-142 bp/yr; the headline is ~5 bp rich.


**The finding survives its anchors, and the headline is slightly rich.** Trimming three months off each end still gives 135.6 bp/yr at *t* = +6.06; using all 618 sessions as anchors gives 141.3 bp/yr, and the median of the same slope across the nine cheap legs is 140.1. The extra ~5 bp in the headline is the tail of GBTC's discount closing in early 2024 — the same thing the era cut shows in §4. **Carry away ~140 bp/yr, not 145.8.** (The trend slope's own HAC *t* is not quoted: its residual is a persistent AR(1) premium, acf1 = 0.80, so that *t* is over-confident. It is used here only as a point estimate that no single day can move.)

## 4. Era cut (split 2025-01-01)

In [6]:
print(f"2024      (n={R['era_e_n']:2d} months): {R['era_e']:+7.1f} bp/yr  "
      f"t={R['era_e_t']:+5.2f}  {R['era_e_pos']}/{R['era_e_n']} positive")
print(f"2025-2026 (n={R['era_l_n']:2d} months): {R['era_l']:+7.1f} bp/yr  "
      f"t={R['era_l_t']:+5.2f}  {R['era_l_pos']}/{R['era_l_n']} positive")

2024      (n=11 months):  +186.5 bp/yr  t=+5.76  9/11 positive
2025-2026 (n=18 months):  +120.9 bp/yr  t=+9.10  17/18 positive


Positive and significant in **both** halves. 2024 runs hot (186 bp/yr) — GBTC's discount was still closing and its outflow was heaviest — and the later era, at 121 bp/yr, is the cleaner steady-state read. Note the *t* is **higher** in the calmer era: as the discount noise faded the contractual fee showed through more clearly, not less.

## 5. Does the fee *ranking* predict the outcome ranking?

Spearman(fee, cohort-relative TD) against a permutation null (exact for ≤ 8 funds, 20,000 sampled above), reported alongside the **attainable** 5% critical value — a fee vector with five funds tied at 25 bp cannot generate a fine rank statistic. Beside it, the cross-sectional pass-through regression TD = a + b·fee, where a wrapper that leaks its fee and nothing else gives b = −1.

In [7]:
print('%-24s %8s %8s %10s %8s %7s' % ('sheet / universe', 'rho', 'p_perm', 'needs|rho|', 'slope b', 'R2'))
for tag, (rho, p, crit, slope, r2) in R['rank'].items():
    print('%-24s %+8.3f %8.4f %10.3f %+8.3f %7.3f' % (tag, rho, p, crit, slope, r2))
print()
print(f"inside the cheap tier the whole fee spread is {R['tier_spread']:.0f} bp/yr;"
      f" the floor is {R['det_vs_peer']:.0f} bp/yr.")

sheet / universe              rho   p_perm needs|rho|  slope b      R2
headline / all ten         -0.642   0.0514      0.642   -1.124   0.978
headline / cheap nine      -0.495   0.1792      0.688   -1.522   0.247
blended / all ten          -0.498   0.1472      0.644   -1.083   0.985
blended / cheap nine       -0.310   0.4076      0.678   -1.645   0.552

inside the cheap tier the whole fee spread is 6 bp/yr; the floor is 66 bp/yr.


Two facts, both true, neither allowed to swallow the other.

- **The pass-through is essentially exact**: slope -1.124 (waiver-blended -1.083) with R² 0.978. The published fee sheet explains 98% of the cross-section.
- **The rank test does not clear — it has no room to**: *p* = 0.0514 against an attainable critical value of 0.642. Drop GBTC and nothing is significant on any fee sheet (*p* = 0.18 to 0.41), and R² falls to 0.247.

This is a power statement, not a contradiction: the cross-section contains exactly **one** resolvable fee fact, and the regression finds it while the rank statistic asks a finer question than 29 months can answer.

> 💡 **In plain words.** The fee sheet is right about who is expensive. It is not, and cannot be, right about who is 22 bp versus 25 bp — that difference is thinner than the measuring instrument.

## 6. The waiver event study

Step in each fund's cohort-relative monthly TD at its **scheduled** waiver end (itself an ASSUMPTION — several were AUM-capped and ended early). A waiver that expires should print a negative step of roughly the headline fee.

In [8]:
print('%-6s %-12s %8s %8s %8s %8s %9s' % ('fund','waiver end','pre','post','step','welch t','expected'))
wrong = 0
for tk, (end, pre, post, step, t, exp) in R['waiver'].items():
    wrong += int(step > 0)
    print('%-6s %-12s %+8.1f %+8.1f %+8.1f %+8.2f %+9d' % (tk, end, pre, post, step, t, exp))
print(f'\nsteps with the WRONG sign: {wrong}/{len(R["waiver"])}; '
      f'max |welch t| = {max(abs(v[4]) for v in R["waiver"].values()):.2f}')

fund   waiver end        pre     post     step  welch t  expected
IBIT   2025-01-10      -20.9    +21.1    +42.0    +0.73       -13
FBTC   2024-07-31      +18.3     +5.0    -13.3    -0.30       -25
ARKB   2024-07-11      +34.6     +6.6    -28.0    -0.22       -21
BITB   2024-07-11       +4.0    +13.2     +9.2    +0.18       -20
HODL   2025-03-31      +28.9    +37.2     +8.3    +0.25       -20
BTCO   2024-07-11      -23.8    +27.5    +51.2    +0.47       -25
EZBC   2024-08-02      +26.1    +15.0    -11.1    -0.11       -19
BTCW   2024-07-11      +19.0     +9.6     -9.3    -0.17       -25

steps with the WRONG sign: 4/8; max |welch t| = 0.73


**Nothing.** No step clears |*t*| = 0.8 and half have the wrong sign. A 20 bp/yr waiver expiry is a **1.7 bp step in a monthly series with a 12 bp sd**, observed with six to twelve months on each side — the test is under-powered by roughly an order of magnitude before it starts. BRRR and GBTC cannot be tested at all (too few months on one side). This is a null result about the *instrument*, not about the waivers.

## 7. Out-of-cohort control — the same sponsor, two fees

Grayscale's Bitcoin Mini Trust (ticker BTC, 15 bp) launched 2024-07-31, spun out of GBTC's own coins. Same sponsor, custodian, coin and strike; 135 bp of fee difference and nothing else.

In [9]:
print(f"BTC - GBTC ({R['mini_months']} months): {R['mini_vs_gbtc']:+7.1f} bp/yr  "
      f"t={R['mini_vs_gbtc_t']:+5.2f}  {R['mini_vs_gbtc_pos']}/{R['mini_months']} positive")
print(f"BTC - {R['cheap']} ({R['mini_months']} months): {R['mini_vs_cheap']:+7.1f} bp/yr  "
      f"t={R['mini_vs_cheap_t']:+5.2f}")
print(f"\nfee difference: {R['mini_fee_gap']} bp/yr. measured: {R['mini_vs_gbtc']:.1f}.")

BTC - GBTC (23 months):  +130.6 bp/yr  t=+2.84  20/23 positive
BTC - EZBC (23 months):    -0.0 bp/yr  t=-0.00

fee difference: 135 bp/yr. measured: 130.6.


Against its own flagship the cheap twin gains the fee. Against an outside cheap wrapper it is indistinguishable from zero. Sponsor identity is held fixed and the effect survives — it is the fee, not the firm.

## 8. The ownership race, excess-of-cash (2 bp one-way, one execution lag)

Three ways to choose a wrapper. `own_cheapest` and `own_priciest` never trade. `rotate_winner` ranks on trailing three-month cohort-relative TD at each quarter end and switches at the **next** session's close. No short leg, so no borrow; the cash leg is BIL's actual total return.

In [10]:
print('%-16s %14s %8s %8s %10s' % ('arm','excess Sharpe','CAGR','vol','total'))
print('%-16s %+14.4f %+8.2f%% %8.1f%% %+10.2f%%'
      % ('own_cheapest', R['sh_cheap'], R['cagr_cheap'], R['vol_ann'], R['tot_cheap']))
print('%-16s %+14.4f %+8.2f%% %8.1f%% %+10.2f%%'
      % ('own_priciest', R['sh_dear'], R['cagr_dear'], R['vol_ann'], R['tot_dear']))
print('%-16s %+14.4f %+8.2f%% %8.1f%% %+10.2f%%'
      % ('rotate_winner', R['sh_rot'], R['cagr_rot'], R['vol_ann'], R['tot_rot']))
print(f"\nSharpe gap cheapest - priciest: {R['sharpe_gap']:+.4f}")
print(f"rotation: {R['race_switches']} switches, and it ends BEHIND doing nothing "
      f"({R['tot_rot']:+.2f}% vs {R['tot_cheap']:+.2f}%)")

arm               excess Sharpe     CAGR      vol      total
own_cheapest            +0.3467    +9.67%     49.9%     +25.36%
own_priciest            +0.3365    +9.13%     49.9%     +23.84%
rotate_winner           +0.3457    +9.64%     49.9%     +25.28%

Sharpe gap cheapest - priciest: +0.0102
rotation: 7 switches, and it ends BEHIND doing nothing (+25.28% vs +25.36%)


The most statistically certain result on this page is worth **+0.0102** of Sharpe, because the asset carries 50% annualised volatility and the fee is 1.2 bp a month. The rotation rule pays 7 switches to arrive behind where buy-and-never-trade already was.

## 9. Costs, tax and borrow — the three ways to act

One-way × NAV throughout. The tax rate, the embedded gain and the borrow rate are **ASSUMPTIONS** (no free tape carries them), so each is swept rather than picked.

In [11]:
print('(1) switch cost - repaid in:')
for c, d in [(2.0, R['be_2bp']), (5.0, R['be_5bp']), (25.0, R['be_25bp'])]:
    print(f'      {c:5.1f} bp one-way -> {d:6.1f} days')
print('\n(2) taxable switch - repaid in:')
print(f'      20.0%  tax, +100% gain -> {R["tax_20_100"]:5.1f} years')
print(f'      20.0%  tax, +300% gain -> {R["tax_20_300"]:5.1f} years')
print(f'      23.8%  tax, +300% gain -> {R["tax_238_300"]:5.1f} years')
print('\n(3) long cheap / short GBTC, net of borrow:')
for b in (0.0, 50.0, 100.0, 150.0, 300.0):
    net = R['ls_gross'] - b
    print(f'      borrow {b:6.1f} bp/yr -> net {net:+8.1f} bp/yr  '
          f'{"alive" if net > 0 else "DEAD"}')

(1) switch cost - repaid in:
        2.0 bp one-way ->   10.0 days
        5.0 bp one-way ->   25.0 days
       25.0 bp one-way ->  125.2 days

(2) taxable switch - repaid in:
      20.0%  tax, +100% gain ->   7.2 years
      20.0%  tax, +300% gain ->  11.1 years
      23.8%  tax, +300% gain ->  13.5 years

(3) long cheap / short GBTC, net of borrow:
      borrow    0.0 bp/yr -> net   +137.8 bp/yr  alive
      borrow   50.0 bp/yr -> net    +87.8 bp/yr  alive
      borrow  100.0 bp/yr -> net    +37.8 bp/yr  alive
      borrow  150.0 bp/yr -> net    -12.2 bp/yr  DEAD
      borrow  300.0 bp/yr -> net   -162.2 bp/yr  DEAD


The long/short survives to roughly **138 bp/yr of borrow** and dies above it — on a permanently hard-to-borrow legacy trust in continuous outflow. The desk does not own a borrow tape, so the honest statement is that whether this nets anything is decided entirely by a number the study cannot observe.

> 💡 **In plain words.** You can prove the expensive fund leaks 135 bp a year. Getting paid for knowing it requires either buying fresh (free), or shorting a stock that is expensive to borrow (unknown), or selling something you owe tax on (a decade).

## 10. Live synthetic control — never supports the stamp

A planted world where each wrapper really is shaved by its published fee, and a null world where every wrapper charges the cohort average while the published sheet still shows the same dispersion. The pair spread and the pass-through must recover the first and stay silent on the second.

In [12]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from fee_war import data, strategy as st
for ss, tag in [(1.0, 'planted fee ladder'), (0.0, 'null: one fee, dispersed sheet')]:
    px, truth = data.synthetic_panel(signal_strength=ss, seed=959)
    d = st.synthetic_detect(px, truth, n_perm=3000)
    print('%-32s pair %+7.1f (planted %3.0f) t=%+6.2f | rho %+.3f p=%.3f | b=%+.3f R2=%.3f'
          % (tag, d['pair_spread_bpy'], d['planted_gap_bpy'], d['pair_t'],
             d['spearman'], d['p_perm'], d['pass_through_slope'], d['pass_through_r2']))
ts, ps = [], []
for s in range(12):
    px, truth = data.synthetic_panel(signal_strength=0.0, seed=959 + s)
    d = st.synthetic_detect(px, truth, n_perm=1500)
    ts.append(abs(d['pair_t'])); ps.append(d['p_perm'])
print('\nnull x12: max |pair t| %.2f (never >= 2), rank test fires at 5%% on %d/12'
      % (max(ts), sum(1 for p in ps if p < 0.05)))

planted fee ladder               pair  +124.4 (planted 131) t= +6.55 | rho -0.545 p=0.119 | b=-0.970 R2=0.995


null: one fee, dispersed sheet   pair    -7.9 (planted   0) t= -0.44 | rho +0.487 p=0.165 | b=+0.039 R2=0.243



null x12: max |pair t| 0.76 (never >= 2), rank test fires at 5% on 3/12


In [13]:
print(f"frozen synthetic run (docs/results.md):")
print(f"  planted: pair {R['syn_pl_pair']:+.1f} bp/yr (planted {R['syn_pl_planted']:.0f}), "
      f"t={R['syn_pl_t']:+.2f}, pass-through b={R['syn_pl_slope']:+.3f} R2={R['syn_pl_r2']:.3f}")
print(f"  null   : pair {R['syn_nl_pair']:+.1f} bp/yr, t={R['syn_nl_t']:+.2f}, "
      f"b={R['syn_nl_slope']:+.3f}")
print(f"  null size, {R['syn_null_seeds']} seeds: mean rho {R['syn_null_rho']:+.3f}, "
      f"rank test fires on {R['syn_null_fire_pct']:.1f}% (nominal 5%), "
      f"max |pair t| {R['syn_null_max_t']:.2f}")
print(f"  power,     {R['syn_pow_seeds']} seeds: mean pair t {R['syn_pow_t']:+.2f}, "
      f"slope {R['syn_pow_slope']:+.3f} +/- {R['syn_pow_slope_sd']:.3f}, "
      f"rank test fires on {R['syn_pow_rank_fire']}/{R['syn_pow_seeds']}")

frozen synthetic run (docs/results.md):
  planted: pair +124.4 bp/yr (planted 131), t=+6.55, pass-through b=-0.970 R2=0.995
  null   : pair -7.9 bp/yr, t=-0.44, b=+0.039
  null size, 120 seeds: mean rho +0.016, rank test fires on 4.2% (nominal 5%), max |pair t| 0.92
  power,     24 seeds: mean pair t +7.00, slope -1.010 +/- 0.045, rank test fires on 9/24


The harness reproduces, on a planted world, **exactly the split the real tape shows**: the pair spread and the pass-through regression see the fee (mean *t* +7.00, slope -1.010 ± 0.045), while the rank statistic — handed a sheet with five funds tied at one number — fires on only 9/24 seeds. On the null it is correctly sized (4.2% at nominal 5%) and the pair instrument's |*t*| never once reaches 2 across 120 seeds. The rank test's failure on the real tape is a property of tied fee sheets, not a broken estimator.

## Verdict

- **Signal — Real.** The headline is a pure-tape fund-versus-fund spread using no fee input: **+145.8 bp/yr, HAC *t* = +8.43**, bootstrap CI [+110.5, +187.0] entirely above zero, 26/29 months positive, significant in **both** eras (186 then 121 bp/yr), replicated on **nine** independent cheap legs (*t* = 5.8 to 10.6). Cross-sectional pass-through of fee into tracking difference: **-1.124, R² = 0.978**. The sponsor-fixed control (Grayscale's own 15 bp twin vs its 150 bp flagship) returns **+130.6 bp/yr** against a 135 bp fee gap.
- **What is not real.** The fine-grained ranking (cheap-tier *p* = 0.18–0.41; 6 bp of fee spread against a 66 bp/yr floor) and the waiver expiries (no step clears |*t*| = 0.8, four of eight the wrong sign).
- **Tradability — Fragile.** +0.0102 of Sharpe against 50% vol; the rotation rule loses to doing nothing; the long/short dies above ~138 bp/yr of unobservable borrow; a taxed switch takes 7–14 years. Bankable only as a **purchase decision**: buy the 20 bp wrapper, never trade it.
- **Sample.** 29 months, one bitcoin cycle, one cohort. The fee arithmetic will not change; everything else should be re-read in five years.